<a href="https://colab.research.google.com/github/AshutoshPant746/CyberSecurity/blob/gh-pages/cyber_lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
pip install scapy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.5 MB/s eta 0:00:00


In [10]:

import pyshark
import re
from collections import Counter
import nest_asyncio

nest_asyncio.apply()
async def extract_email_addresses(pcap_file):
    senders = []
    receivers = []
    sender_receiver_pairs = []

    capture = pyshark.FileCapture(pcap_file, display_filter='smtp or imf', use_json=True)
    email_pattern = re.compile(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+')

    for packet in capture:
        try:
            if hasattr(packet, 'smtp') or hasattr(packet, 'imf'):
                smtp_payload = str(packet)
                emails = email_pattern.findall(smtp_payload)
                if len(emails) >= 2:
                    sender = emails[0]
                    receiver = emails[1]
                    senders.append(sender)
                    receivers.append(receiver)
                    sender_receiver_pairs.append((sender, receiver))
        except Exception as e:
            print(f"Error processing packet: {e}")

    capture.close()
    return senders, receivers, sender_receiver_pairs

async def analyze_email_traffic_async(pcap_file):
    senders, receivers, sender_receiver_pairs = await extract_email_addresses(pcap_file)

    sender_counts = Counter(senders)
    receiver_counts = Counter(receivers)
    pair_counts = Counter(sender_receiver_pairs)

    print("Top 5 Most Frequent Sender Email Addresses:")
    for sender, count in sender_counts.most_common(5):
        print(f"{sender}: {count} times")

    print("\nTop 5 Most Frequent Receiver Email Addresses:")
    for receiver, count in receiver_counts.most_common(5):
        print(f"{receiver}: {count} times")

    print("\nTop 5 Most Frequent Sender-Receiver Pairs:")
    for pair, count in pair_counts.most_common(5):
        print(f"{pair[0]} -> {pair[1]}: {count} times")

pcap_file = "/content/drive/MyDrive/Colab Notebooks/fake_email_traffic.pcap"
await analyze_email_traffic_async(pcap_file)

Top 5 Most Frequent Sender Email Addresses:
sender829@example.com: 5 times
sender335@example.com: 4 times
sender323@example.com: 4 times
sender233@example.com: 4 times
sender325@example.com: 4 times

Top 5 Most Frequent Receiver Email Addresses:
receiver837@example.com: 5 times
receiver853@example.com: 5 times
receiver492@example.com: 5 times
receiver57@example.com: 5 times
receiver642@example.com: 5 times

Top 5 Most Frequent Sender-Receiver Pairs:
sender108@example.com -> receiver386@example.com: 1 times
sender324@example.com -> receiver218@example.com: 1 times
sender644@example.com -> receiver733@example.com: 1 times
sender187@example.com -> receiver837@example.com: 1 times
sender50@example.com -> receiver309@example.com: 1 times


In [14]:
import scapy.all as scapy
from collections import Counter


def extract_smtp_commands(pcap_file):
    packets = scapy.rdpcap(pcap_file)

    smtp_commands = ['EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT']
    command_counts = {command: 0 for command in smtp_commands}
    command_sequence = []

    for packet in packets:
        if packet.haslayer(scapy.Raw):
            raw_data = packet[scapy.Raw].load.decode(errors='ignore')

            for command in smtp_commands:
                if command in raw_data:
                    command_counts[command] += 1
                    command_sequence.append(command)

    return command_counts, command_sequence


def detect_anomalies(command_counts, command_sequence):
    anomalies = []

    expected_sequence = ['EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT']

    current_position = 0
    for i, command in enumerate(command_sequence):
        if current_position < len(expected_sequence):

            if command == expected_sequence[current_position]:
                current_position += 1
            else:
                anomalies.append(f"Anomalous sequence detected at index {i}: {command_sequence}")
                break
        else:

            anomalies.append(f"Extra command detected at index {i}: {command_sequence}")
            break

    return anomalies

pcap_file = "/content/drive/MyDrive/Colab Notebooks/fake_email_traffic.pcap"

command_counts, command_sequence = extract_smtp_commands(pcap_file)

print("SMTP Command Frequency Analysis:")
for command, count in command_counts.items():
    print(f"{command}: {count} times")

anomalies = detect_anomalies(command_counts, command_sequence)

if anomalies:
    print("\nAnomalies Detected:")
    for anomaly in anomalies:
        print(anomaly)
else:
    print("\nNo anomalies detected.")


SMTP Command Frequency Analysis:
EHLO: 999 times
MAIL FROM: 1000 times
RCPT TO: 1000 times
DATA: 1000 times
QUIT: 1000 times

Anomalies Detected:
Extra command detected at index 5: ['EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'EHLO', 'MAIL FROM', 'RCPT TO', 'DATA', 'QUIT', 'E

In [17]:
import scapy.all as scapy
from collections import defaultdict


def extract_ip_addresses(pcap_file):

    packets = scapy.rdpcap(pcap_file)


    source_ips = defaultdict(int)
    dest_ips = defaultdict(int)


    recipient_emails = defaultdict(set)


    for packet in packets:
        if packet.haslayer(scapy.IP) and packet.haslayer(scapy.Raw):

            src_ip = packet[scapy.IP].src
            dest_ip = packet[scapy.IP].dst


            source_ips[src_ip] += 1
            dest_ips[dest_ip] += 1


            raw_data = packet[scapy.Raw].load.decode(errors='ignore')
            if "RCPT TO" in raw_data:

                start = raw_data.find("RCPT TO:<") + 9
                end = raw_data.find(">", start)
                if start != -1 and end != -1:
                    recipient_email = raw_data[start:end]
                    recipient_emails[recipient_email].add(src_ip)

    return source_ips, dest_ips, recipient_emails


def detect_suspicious_behavior(source_ips, dest_ips, recipient_emails):
    suspicious_ips = []
    suspicious_recipients = []


    for src_ip, count in source_ips.items():
        if count > 200:
            suspicious_ips.append((src_ip, count))


    for recipient, ips in recipient_emails.items():
        if len(ips) > 3:
            suspicious_recipients.append((recipient, len(ips), ips))

    return suspicious_ips, suspicious_recipients

pcap_file = "/content/drive/MyDrive/Colab Notebooks/fake_email_traffic.pcap"

source_ips, dest_ips, recipient_emails = extract_ip_addresses(pcap_file)


suspicious_ips, suspicious_recipients = detect_suspicious_behavior(source_ips, dest_ips, recipient_emails)


print("Suspicious Source IPs (more than 200 emails sent):")
for ip, count in suspicious_ips:
    print(f"Source IP: {ip}, Emails Sent: {count}")

print("\nSuspicious Recipients (more than 3 unique source IPs sending emails):")
for recipient, num_ips, ips in suspicious_recipients:
    print(f"Recipient: {recipient}, Source IPs: {num_ips}, IPs: {', '.join(ips)}")

Suspicious Source IPs (more than 200 emails sent):
Source IP: 192.168.1.30, Emails Sent: 208
Source IP: 172.16.0.3, Emails Sent: 231

Suspicious Recipients (more than 3 unique source IPs sending emails):
Recipient: mple.com
MAIL FROM: <sender829@example.com, Source IPs: 4, IPs: 192.168.1.20, 192.168.1.30, 10.0.0.5, 172.16.0.3
